In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Adatok betöltése
df = pd.read_csv('zenga_listings_details_optimized_timi.csv')

# Előkészítés: oszlopok tisztítása
def clean_numeric_column(column):
    """Számos oszlop tisztítása"""
    if column.dtype == object:
        return pd.to_numeric(column.str.replace(' Ft/hó', '').str.replace(' ', '').str.replace(',', '.'), errors='coerce')
    return column

# Numerikus oszlopok tisztítása
df['price'] = clean_numeric_column(df['price'])
df['area_m2'] = clean_numeric_column(df['area_m2'])
df['rooms'] = clean_numeric_column(df['rooms'])
df['Átlagos rezsiköltség'] = clean_numeric_column(df['Átlagos rezsiköltség'])
df['Építés éve'] = clean_numeric_column(df['Építés éve'])
df['Erkély'] = clean_numeric_column(df['Erkély'])
df['Terasz'] = clean_numeric_column(df['Terasz'])

# Outlier szűrés funkció
def remove_outliers_iqr(df, column):
    """Outlierek eltávolítása IQR módszerrel"""
    if column not in df.columns:
        return df
    
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Percentilis alapú normalizáció
def percentile_score(value, series, reverse=False):
    """Percentilis alapú pontszám 0-100 skálán"""
    if pd.isna(value):
        return 50
    
    valid_series = series.dropna()
    if len(valid_series) == 0:
        return 50
    
    percentile = stats.percentileofscore(valid_series, value)
    
    if reverse:
        # Minél kisebb érték, annál jobb (pl. ár)
        return 100 - percentile
    else:
        # Minél nagyobb érték, annál jobb (pl. állapot)
        return percentile

# Mutatók számítása
def calculate_indicators(df):
    results = []
    
    # Először gyűjtsük össze az összes értéket percentilisekhez
    all_teljes_koltseg_m2 = []
    all_epites_evek = []
    all_erkely_meretek = []
    
    for _, row in df.iterrows():
        # Teljes költség/m² előkészítése
        rezsi = row['Átlagos rezsiköltség'] if pd.notna(row['Átlagos rezsiköltség']) else 30000
        teljes_koltseg = (row['price'] if pd.notna(row['price']) else 0) + rezsi
        teljes_koltseg_m2 = teljes_koltseg / (row['area_m2'] if pd.notna(row['area_m2']) and row['area_m2'] > 0 else 1)
        all_teljes_koltseg_m2.append(teljes_koltseg_m2)
        
        # Építés éve
        if pd.notna(row['Építés éve']) and row['Építés éve'] > 1800:
            all_epites_evek.append(row['Építés éve'])
        
        # Erkély mérete
        erkely_meret = 0
        if pd.notna(row['Erkély']):
            erkely_meret += row['Erkély']
        if pd.notna(row['Terasz']):
            erkely_meret += row['Terasz']
        all_erkely_meretek.append(erkely_meret)
    
    # Konvertálás Series-é percentilis számításhoz
    teljes_koltseg_series = pd.Series(all_teljes_koltseg_m2)
    epites_eve_series = pd.Series(all_epites_evek)
    erkely_meret_series = pd.Series(all_erkely_meretek)
    
    # Mutatók számítása minden ingatlanra
    for idx, row in df.iterrows():
        indicator_data = {
            'url': row['url'],
            'title': row['title'],
            'price': row['price'],
            'area_m2': row['area_m2'],
            'rooms': row['rooms'],
            'location': row['location']
        }
        
        # 1. TELJES KÖLTSÉG/M² (minél kisebb, annál jobb)
        rezsi = row['Átlagos rezsiköltség'] if pd.notna(row['Átlagos rezsiköltség']) else 30000
        teljes_koltseg = (row['price'] if pd.notna(row['price']) else 0) + rezsi
        teljes_koltseg_m2 = teljes_koltseg / (row['area_m2'] if pd.notna(row['area_m2']) and row['area_m2'] > 0 else 1)
        
        koltseg_m2_score = percentile_score(teljes_koltseg_m2, teljes_koltseg_series, reverse=True)
        indicator_data['teljes_koltseg_m2_score'] = koltseg_m2_score
        indicator_data['teljes_koltseg_m2'] = teljes_koltseg_m2
        
        # 2. ENERGETIKAI BESOROLÁS
        energia_map = {'A++': 100, 'A+': 95, 'A': 90, 'B': 80, 'C': 70, 'D': 60, 
                      'E': 50, 'F': 40, 'G': 30, 'CC (2016-2023-as besorolás)': 65}
        energia_score = energia_map.get(row['Energetikai besorolás'], 50) if pd.notna(row['Energetikai besorolás']) else 50
        indicator_data['energia_score'] = energia_score
        
        # 3. INGATLAN ÁLLAPOTA
        allapot_map = {'Új építésű': 100, 'Újszerű': 90, 'Felújított': 80, 
                      'Jó állapotú': 70, 'Átlagos': 50, 'Felújítandó': 20}
        allapot_score = allapot_map.get(row['Állapot'], 40) if pd.notna(row['Állapot']) else 50
        indicator_data['allapot_score'] = allapot_score
        
        # 4. ÉPÍTÉS ÉVE (minél újabb, annál jobb)
        if pd.notna(row['Építés éve']) and row['Építés éve'] > 1800:
            ev_score = percentile_score(row['Építés éve'], epites_eve_series, reverse=False)
        else:
            ev_score = 50
        indicator_data['epites_eve_score'] = ev_score
        
        # 5. FŰTÉS TÍPUSA
        futes_map = {
            'Hőszivattyú': 100, 'Házközponti fűtés egyedi méréssel': 90, 'Távfűtés egyedi méréssel': 85,
            'Központi fűtés': 80, 'Házközponti': 80, 'Távfűtés': 75, 'Gázkazán': 65,
            'Gáz (konvektor)': 50, 'Gáz (héra)': 55, 'Elektromos': 40, 'Egyéb': 30
        }
        futes_score = futes_map.get(row['Fűtés'], 40) if pd.notna(row['Fűtés']) else 50
        indicator_data['futes_score'] = futes_score
        
        # 6. ERKÉLY/TERASZ (minél nagyobb, annál jobb)
        erkely_meret = 0
        if pd.notna(row['Erkély']):
            erkely_meret += row['Erkély']
        if pd.notna(row['Terasz']):
            erkely_meret += row['Terasz']
        
        erkely_score = percentile_score(erkely_meret, erkely_meret_series, reverse=False)
        indicator_data['erkely_score'] = erkely_score
        indicator_data['erkely_meret'] = erkely_meret
        
        # 7. NAPELEM
        napelem_score = 100 if row['Napelem'] == 'Van' else (50 if row['Napelem'] == 'Nincs' else 0)
        indicator_data['napelem_score'] = napelem_score
        
        results.append(indicator_data)
    
    return pd.DataFrame(results), teljes_koltseg_series, epites_eve_series

# Outlier szűrés a fontos numerikus oszlopokon
print("Outlier szűrés előtt:", len(df))
df_clean = df.copy()
for column in ['price', 'area_m2', 'Átlagos rezsiköltség']:
    df_clean = remove_outliers_iqr(df_clean, column)
print("Outlier szűrés után:", len(df_clean))

# Mutatók kiszámítása
indicators_df, teljes_koltseg_series, epites_eve_series = calculate_indicators(df_clean)

# Súlyozott pontszám számítása
sulyok = {
    'teljes_koltseg_m2_score': 0.95,  # Legfontosabb: teljes költség hatékonyság
    'energia_score': 0.20,            # Energiahatékonyság
    'allapot_score': 0.90,            # Ingatlan állapota
    'epites_eve_score': 0.10,         # Építés éve
    'futes_score': 0.00,              # Fűtés típusa
    'erkely_score': 0.50,             # Erkély/terasz
    'napelem_score': 0.00             # Napelem
}

# Végső pontszám számítása
def calculate_final_score(row):
    score = 0
    for mutato, suly in sulyok.items():
        score += row[mutato] * suly
    return round(score, 1)

indicators_df['vegso_pontszam'] = indicators_df.apply(calculate_final_score, axis=1)

# Eredmények rendezése és megjelenítése
top_ingatlanok = indicators_df.sort_values('vegso_pontszam', ascending=False).head(20)

print("\n" + "="*120)
print("TOP 20 LEGJOBB INGATLAN (PERCENTILIS ALAPÚ SULYOZOTT PONTSZÁM)")
print("="*120)

for idx, (_, ingatlan) in enumerate(top_ingatlanok.iterrows(), 1):
    print(f"\n{idx}. {ingatlan['title']}")
    print(f"   📍 Helyszín: {ingatlan['location']}")
    print(f"   🔗 URL: {ingatlan['url']}")
    print(f"   💰 Ár: {ingatlan['price']:,.0f} Ft, 📐 Terület: {ingatlan['area_m2']} m², 🚪 Szobák: {ingatlan['rooms']}")
    print(f"   📊 Teljes költség/m²: {ingatlan['teljes_koltseg_m2']:,.0f} Ft/m²")
    print(f"   🏆 Végső pontszám: {ingatlan['vegso_pontszam']:.1f}/100")
    print(f"   📈 Részpontszámok: Költség({ingatlan['teljes_koltseg_m2_score']:.1f}) | " +
          f"Energia({ingatlan['energia_score']:.1f}) | Állapot({ingatlan['allapot_score']:.1f}) | " +
          f"Év({ingatlan['epites_eve_score']:.1f}) | Erkély({ingatlan['erkely_score']:.1f})")

# Statisztikák
print(f"\n{'='*60} STATISZTIKÁK {'='*60}")
print(f"📊 Összes ingatlan: {len(indicators_df)}")
print(f"📈 Átlagos pontszám: {indicators_df['vegso_pontszam'].mean():.1f}")
print(f"🥇 Legjobb pontszám: {indicators_df['vegso_pontszam'].max():.1f}")
print(f"📉 Legrosszabb pontszám: {indicators_df['vegso_pontszam'].min():.1f}")

# Mutatók átlagai
print(f"\n📋 Átlagos mutatóértékek:")
for mutato, suly in sulyok.items():
    avg_score = indicators_df[mutato].mean()
    print(f"   {mutato}: {avg_score:.1f} (súly: {suly*100}%)")

# Percentilis statisztikák a fontos mutatókról
print(f"\n📊 Percentilis statisztikák:")
print(f"   Teljes költség/m² - 25%: {teljes_koltseg_series.quantile(0.25):.0f} Ft, " +
      f"50%: {teljes_koltseg_series.quantile(0.5):.0f} Ft, " +
      f"75%: {teljes_koltseg_series.quantile(0.75):.0f} Ft")
print(f"   Építés év - 25%: {epites_eve_series.quantile(0.25):.0f}, " +
      f"50%: {epites_eve_series.quantile(0.5):.0f}, " +
      f"75%: {epites_eve_series.quantile(0.75):.0f}")